In [ ]:
import pandas as pd
import pyarrow.parquet as pq
from neo4j import GraphDatabase
import time
import os
from dotenv import load_dotenv

# ==========================================
# 1. CONFIGURACIÓN
# ==========================================
load_dotenv() 

URI = os.getenv("NEO4J_URI")
USER = os.getenv("NEO4J_USER")
PASSWORD = os.getenv("NEO4J_PASSWORD")
#PARQUET_FILE = 'part-00000-5f556208-a1fc-40a1-9cc2-a4e24c76aeb3-c000.snappy.parquet'
#PARQUET_FILE = 'part-00000-8b838a85-76eb-4896-a0b6-2fc425e828c2-c000.snappy.parquet'
#PARQUET_FILE = 'part-00000-0955ed97-8460-41bd-872a-7375a7f0207e-c000.snappy.parquet'
#PARQUET_FILE = 'part-00000-69700ccb-c1c1-4763-beb7-cd0f1a61c268-c000.snappy.parquet'
#PARQUET_FILE = 'part-00000-071774ae-97f3-4f31-9700-8bfcdf41305a-c000.snappy.parquet'
#PARQUET_FILE = 'part-00000-ea3a47a3-0973-4d6b-a3a2-8dd441ee7901-c000.snappy.parquet'
PARQUET_FILE = 'part-00000-f078acc1-ab56-40a6-a6e1-99d780645c57-c000.snappy.parquet'
BATCH_SIZE = 10000

# ==========================================
# 2. LA QUERY CYPHER (ANTI-DUPLICADOS)
# ==========================================
query = """
UNWIND $batch AS row

// --- NODO ORIGEN ---
MERGE (src:IP {address: row.src_ip_zeek})
ON CREATE SET 
    src.conns_started = 1, 
    src.total_bytes_sent = row.orig_bytes,
    src.total_pkts_sent = row.orig_pkts
ON MATCH SET 
    src.conns_started = coalesce(src.conns_started, 0) + 1, 
    src.total_bytes_sent = coalesce(src.total_bytes_sent, 0) + row.orig_bytes,
    src.total_pkts_sent = coalesce(src.total_pkts_sent, 0) + row.orig_pkts

// --- NODO DESTINO ---
MERGE (dst:IP {address: row.dest_ip_zeek})
ON CREATE SET 
    dst.conns_received = 1, 
    dst.total_bytes_received = row.resp_bytes,
    dst.total_pkts_received = row.resp_pkts
ON MATCH SET 
    dst.conns_received = coalesce(dst.conns_received, 0) + 1, 
    dst.total_bytes_received = coalesce(dst.total_bytes_received, 0) + row.resp_bytes,
    dst.total_pkts_received = coalesce(dst.total_pkts_received, 0) + row.resp_pkts

// --- LA RELACIÓN (Bloqueada por UID para no duplicar) ---
MERGE (src)-[c:CONNECTED_TO {uid: row.uid}]->(dst)
ON CREATE SET
    c.ts = row.ts,
    c.src_port = toInteger(row.src_port_zeek),
    c.dst_port = toInteger(row.dest_port_zeek),
    c.protocol = row.proto,
    c.service = row.service,
    c.duration = toFloat(row.duration),
    c.conn_state = row.conn_state,
    c.is_malicious = row.label_binary,
    c.mitre_tactic = row.label_tactic,
    c.mitre_technique = row.label_technique,
    c.cve = row.label_cve
"""

def process_batch(tx, batch):
    tx.run(query, batch=batch)

# ==========================================
# 3. PROCESAMIENTO EN STREAMING
# ==========================================
print(f"Abriendo {PARQUET_FILE} para procesamiento en lotes...")
parquet_file = pq.ParquetFile(PARQUET_FILE)

# Conectamos a Neo4j
driver = GraphDatabase.driver(URI, auth=(USER, PASSWORD)) # type: ignore

start_time = time.time()
batch_count = 0
total_rows = 0

with driver.session() as session:
    # iter_batches lee el archivo en fragmentos sin cargar todo en RAM
    for batch in parquet_file.iter_batches(batch_size=BATCH_SIZE):
        batch_count += 1
        
        # Convertimos el batch de PyArrow a Pandas DataFrame
        df = batch.to_pandas()
        
        # Limpieza rápida de datos numéricos
        columnas_numericas = ['duration', 'orig_bytes', 'resp_bytes', 'orig_pkts', 'resp_pkts', 'missed_bytes']
        for col in columnas_numericas:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

        # Limpieza de textos nulos (y asegurarnos de que el uid no falle)
        columnas_texto = ['uid', 'service', 'conn_state', 'label_tactic', 'label_technique', 'label_binary', 'label_cve']
        for col in columnas_texto:
            df[col] = df[col].fillna('N/A')

        # Convertimos a diccionario
        batch_data = df.to_dict('records')
        
        # Insertamos en Neo4j
        session.execute_write(process_batch, batch_data)
        
        total_rows += len(batch_data)
        print(f"Lote {batch_count} procesado. Filas totales insertadas: {total_rows}...")

driver.close()
end_time = time.time()

print(f"\n¡Carga Masiva Completada!")
print(f"Total de registros: {total_rows}")
print(f"Tiempo transcurrido: {round((end_time - start_time)/60, 2)} minutos.")

Abriendo part-00000-f078acc1-ab56-40a6-a6e1-99d780645c57-c000.snappy.parquet para procesamiento en lotes...
Lote 1 procesado. Filas totales insertadas: 10000...
Lote 2 procesado. Filas totales insertadas: 20000...
Lote 3 procesado. Filas totales insertadas: 30000...
Lote 4 procesado. Filas totales insertadas: 40000...
Lote 5 procesado. Filas totales insertadas: 50000...
Lote 6 procesado. Filas totales insertadas: 60000...
Lote 7 procesado. Filas totales insertadas: 70000...
Lote 8 procesado. Filas totales insertadas: 80000...
Lote 9 procesado. Filas totales insertadas: 90000...
Lote 10 procesado. Filas totales insertadas: 100000...
Lote 11 procesado. Filas totales insertadas: 110000...
Lote 12 procesado. Filas totales insertadas: 120000...
Lote 13 procesado. Filas totales insertadas: 130000...
Lote 14 procesado. Filas totales insertadas: 140000...
Lote 15 procesado. Filas totales insertadas: 150000...
Lote 16 procesado. Filas totales insertadas: 160000...
Lote 17 procesado. Filas total